# Vision Transformer Image Classification

This notebook implements an image classification pipeline with a Vision Transformer (ViT) using `torchvision`. The workflow covers device selection, image preprocessing, dataset construction, train/test splitting, batching, training a ViT from scratch, evaluating it, and fine-tuning a pretrained ViT.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.models.vision_transformer import VisionTransformer
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from PIL import Image
import os
import matplotlib.pyplot as plt

## 1. Compute Device

Select the available computation device. When CUDA is available, the model and tensors can be moved to the GPU to accelerate training and evaluation.

In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

device(type='cuda')

## 2. Dataset Location

Define the root directory containing the image dataset. The dataset is organized in class-specific folders, so each folder name can be used as a classification label.

In [3]:
DATA_DIR = './dataset/images'

## 3. Image Preprocessing

Before entering the Vision Transformer, every image is resized to **224 × 224** and converted into a PyTorch tensor.

**ViT design note:** with a `16 × 16` patch size, a `224 × 224` image produces `14 × 14 = 196` image patches. Therefore, the image dimensions should be compatible with the selected patch size.

In [4]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

## 4. Build the ImageFolder Dataset

Create the dataset with `ImageFolder`. Each subdirectory represents one class, while the images inside that directory inherit the corresponding class label. The previously defined transform is applied when samples are loaded.

In [5]:
full_dataset = datasets.ImageFolder(DATA_DIR, transform=transform)
train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size

### Training Set Size

Use 80% of the available images for training. The resulting value is converted to an integer because a dataset size must be a whole number.

In [6]:
train_size

1626

### Test Set Size

Assign the remaining 20% of the images to the test set. Keeping the test data separate allows the final model performance to be measured on samples that were not used for parameter updates.

In [7]:
test_size

407

## 5. Train/Test Split

Randomly split the complete dataset into training and test subsets using the calculated sizes. This creates the two datasets used by the training and evaluation stages.

In [8]:
train_db, test_db = torch.utils.data.random_split(full_dataset, [train_size, test_size])

## 6. DataLoaders and Batching

Wrap the training and test subsets in `DataLoader` objects so the model receives data in manageable mini-batches.

- `batch_size=8` controls how many images are processed at once.
- `shuffle=True` randomizes the training order between iterations, reducing dependence on the original sample order.
- `shuffle=False` keeps test evaluation deterministic and preserves the test subset order.

In [9]:
BATCH_SIZE = 8
train_loader = DataLoader(train_db, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_db, batch_size=BATCH_SIZE, shuffle=False)

## 7. Class Labels

Retrieve the class names directly from `ImageFolder`. These labels define the output categories used by the classifier.

In [10]:
class_names = full_dataset.classes # ['airplane', 'face', 'motorcycle']

# 8. Vision Transformer from Scratch

The first experiment constructs a Vision Transformer directly with configurable architectural parameters. The selected configuration follows the ViT-Base-style setup discussed in the accompanying material.

# model_scratch

### Model Configuration

The architecture uses a `224 × 224` input, `16 × 16` patches, 12 Transformer layers, 12 attention heads, a 768-dimensional hidden representation, and a 3072-dimensional MLP representation. The final number of output classes is matched to the dataset.

In [11]:
NUM_CLASSES = 3

model_scratch = VisionTransformer(
    image_size=224,
    patch_size=16,
    num_layers=12,
    num_heads=12,
    hidden_dim=768,
    mlp_dim=3072,
    num_classes=NUM_CLASSES
)

### Move the Model to the Selected Device

Transfer the model to the same device selected earlier so that model parameters and input tensors can be processed consistently on the CPU or GPU.

In [12]:
model_scratch = model_scratch.to(DEVICE)

### Loss Function and Optimizer

Use cross-entropy loss for multi-class classification and Adam to update the model parameters during training. The learning rate controls the size of each optimization step.

In [13]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_scratch.parameters(), lr=1e-4)

## 9. Train the ViT from Scratch

Run the training loop for the configured number of epochs. For each mini-batch, the model performs a forward pass, the classification loss is computed, gradients are backpropagated, and the optimizer updates the parameters.

In [14]:
EPOCHS = 3

for epoch in range(EPOCHS):
    model_scratch.train()
    total_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model_scratch(images) 
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    print(f"Scratch - Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss/len(train_loader):.4f}")

Scratch - Epoch 1/3, Loss: 0.8316
Scratch - Epoch 2/3, Loss: 0.5116
Scratch - Epoch 3/3, Loss: 0.3395


## 10. Evaluate the Scratch Model

Switch the model to evaluation mode and disable gradient tracking. Predictions on the held-out test set are compared with the true labels to calculate classification accuracy.

In [15]:
model_scratch.eval()
correct = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model_scratch(images) 
        _, pred = torch.max(outputs, 1)
        correct += (pred == labels).sum().item()

total_acc = 100 * correct / len(test_db)
print(f"model_scratch: {total_acc:.2f}%")

model_scratch: 89.68%


# 11. Pretrained Vision Transformer

The second experiment starts from a Vision Transformer pretrained on ImageNet. Using pretrained representations can provide a stronger initialization when the target dataset is relatively small.

# model_pretrained

### Load pretrained ViT-B/16 Weights

Load the `ViT-B/16` architecture with ImageNet-1K pretrained weights from `torchvision`. The `/16` notation refers to the `16 × 16` patch size.

In [16]:
model_pretrained = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)

### Adapt the Classification Head

The pretrained model was originally designed for ImageNet classification. Replace its final classification layer so that its output dimension matches the number of classes in the current dataset.

In [17]:
model_pretrained.heads.head = nn.Linear(model_pretrained.heads.head.in_features, NUM_CLASSES)

### Move the Pretrained Model to the Device

Place the adapted pretrained model on the selected computation device before training.

In [18]:
model_pretrained = model_pretrained.to(DEVICE)

### Fine-Tuning Objective

Use the same cross-entropy classification objective and Adam optimizer for fine-tuning the pretrained model on the target dataset.

In [19]:
criterion = nn.CrossEntropyLoss()
optimizer_pt = optim.Adam(model_pretrained.parameters(), lr=1e-4)

### Confirm Device Placement

Keep the pretrained model on the selected device before entering the fine-tuning loop.

In [20]:
model_pretrained = model_pretrained.to(DEVICE)

## 12. Fine-Tune the Pretrained Model

Fine-tune the pretrained representation on the target training set. The model can reuse useful visual features learned from the large-scale pretraining dataset while adapting its parameters to the new classes.

In [21]:
for epoch in range(EPOCHS):
    model_pretrained.train()
    total_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        
        optimizer_pt.zero_grad() 
        outputs = model_pretrained(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_pt.step()
        total_loss += loss.item()
    
    print(f"Pretrained - Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss/len(train_loader):.4f}")

Pretrained - Epoch 1/3, Loss: 0.0479
Pretrained - Epoch 2/3, Loss: 0.0004
Pretrained - Epoch 3/3, Loss: 0.0001


## 13. Evaluate the Fine-Tuned Model

Evaluate the fine-tuned Vision Transformer on the held-out test set using the same accuracy procedure. Comparing this result with the from-scratch model provides a direct view of the benefit of pretrained initialization.

In [22]:
model_pretrained.eval()
correct = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model_pretrained(images)
        _, pred = torch.max(outputs, 1)
        correct += (pred == labels).sum().item()

total_acc = 100 * correct / len(test_db)
print(f"model_pretrained: {total_acc:.2f}%")

model_pretrained: 100.00%


# Summary

This notebook compares two ViT strategies for image classification: training a configurable Vision Transformer from scratch and fine-tuning a pretrained ViT-B/16. The comparison highlights how dataset scale and pretrained representations can influence Vision Transformer performance.